In [27]:
# ---------------------------------------
# Mount Google Drive
# ---------------------------------------
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [28]:
# ---------------------------------------
# step 2: importing the required libraries
# ---------------------------------------


import os
import random
import numpy as np
import pandas as pd
import scipy.io as sio
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, train_test_split

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, DepthwiseConv2D
from tensorflow.keras.layers import AveragePooling2D, SeparableConv2D
from tensorflow.keras.layers import BatchNormalization, Activation
from tensorflow.keras.layers import Dropout, Flatten, Dense
from tensorflow.keras.constraints import max_norm
from tensorflow.keras.utils import to_categorical

In [29]:
# ---------------------------------------
# step 3 : Basic experiment settings
# ---------------------------------------


random_seed = 42
subject_file_name = "S14_EEG.mat"

dataset_folder = "/content/drive/MyDrive/Colab_Notebooks/My_Project/imagined_speech/Base de Datos Habla Imaginada/S14"
results_folder = "/content/drive/MyDrive/Colab_Notebooks/My_Project/Results_April2026"

number_of_signal_columns = 24576
number_of_channels = 6
number_of_samples_per_trial = 4096
number_of_folds = 5

batch_size = 16
number_of_epochs = 50
validation_size_inside_training = 0.2

In [30]:
# ---------------------------------------
# step 4 : make results folder and fix randomness
# ---------------------------------------

# Creates the results folder if it doesn't exist
os.makedirs(results_folder, exist_ok=True)

np.random.seed(random_seed)
random.seed(random_seed)
tf.random.set_seed(random_seed)
print("==============================================")
print("Results folder is ready.")
print("Random seed is set.")
print("==============================================")

Results folder is ready.
Random seed is set.


In [26]:
# ---------------------------------------
# step 5 : building the subject path and load the file
# ---------------------------------------

import scipy.io as sio

subject_file_name = "S14_EEG.mat"

subject_file_path = os.path.join(dataset_folder, subject_file_name)

print("==============================================")
print("Subject file path:", subject_file_path)
print("Does file exist?", os.path.exists(subject_file_path))
print("==============================================")
mat_data = sio.loadmat(subject_file_path)
print("==============================================")
print("\nThe .mat file loaded successfully.")
print("Available keys inside the file:", mat_data.keys())
print("==============================================")

Subject file path: /content/drive/MyDrive/Colab_Notebooks/My_Project/imagined_speech/Base de Datos Habla Imaginada/S14/S14_EEG.mat
Does file exist? True

The .mat file loaded successfully.
Available keys inside the file: dict_keys(['__header__', '__version__', '__globals__', 'EEG'])


In [33]:
# ---------------------------------------
# step 6 : extract EEG matrix and printing shape
# ---------------------------------------

eeg_matrix = mat_data["EEG"]
print("==============================================")
print("EEG matrix extracted.")
print("number_of_trials, number_of_columns are : ")
print("==============================================")
print("EEG matrix shape:", eeg_matrix.shape)
print("==============================================")

EEG matrix extracted.
number_of_trials, number_of_columns are : 
EEG matrix shape: (639, 24579)


In [ ]:
"""
EEG matrix shape is: (639, 24579)
That means:
-->> 639 rows ->  most likely trials
-->> 24579 columns ->  this matches earlier understanding:
      24576 signal values
      1 modality column
      1 stimulus/label column
      1 artifact column
So the structure looks consistent.

NEXT STPE IS: Separating the EEG signal and metadata

We split the matrix into 4 parts:
    1) signal data
    2) modality column
    3) stimulus/label column
    4) artifact column

This is important because:
    -> only the signal goes into the model
    -> the other columns help us decide which trials to keep
"""

In [34]:
# ---------------------------------------
# step 7 : separating the signal data and metadata columns
# ---------------------------------------



number_of_signal_columns = 24576

signal_data = eeg_matrix[:, :number_of_signal_columns]
modality_column = eeg_matrix[:, number_of_signal_columns]
stimulus_column = eeg_matrix[:, number_of_signal_columns + 1]
artifact_column = eeg_matrix[:, number_of_signal_columns + 2]

print("==============================================")
print("Signal data shape:", signal_data.shape)
print("Modality column shape:", modality_column.shape)
print("Stimulus column shape:", stimulus_column.shape)
print("Artifact column shape:", artifact_column.shape)
print("==============================================")

print("Unique modality values:", np.unique(modality_column))
print("Unique stimulus values:", np.unique(stimulus_column))
print("Unique artifact values:", np.unique(artifact_column))
print("==============================================")


Signal data shape: (639, 24576)
Modality column shape: (639,)
Stimulus column shape: (639,)
Artifact column shape: (639,)
Unique modality values: [1. 2.]
Unique stimulus values: [ 1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11.]
Unique artifact values: [1. 2.]


In [ ]:
"""
number_of_signal_columns = 24576 -->> that means, the first 24576 columns are real EEG signal values
6 channels and 4096 samples per channel
6 × 4096 = 24576

So each row contains one complete EEG trial flattened into one long row.

signal_data = eeg_matrix[:, :number_of_signal_columns]
this means: take all rows and take columns from start up to 24576 -->> This gives us only the EEG signal part.

modality_column = eeg_matrix[:, number_of_signal_columns]
This picks the next column after the signal.
This column tells which modality the trial belongs to.
Later we will use it to keep only imagined speech trials.

stimulus_column = eeg_matrix[:, number_of_signal_columns + 1]
This picks the next column after modality.
This is the class label or stimulus label.
This tells the target category for each trial.

artifact_column = eeg_matrix[:, number_of_signal_columns + 2]
This picks the final metadata column.
This tells whether the trial is clean or artifact-related according to the dataset coding.

np.unique(modality_column)
np.unique(stimulus_column)
np.unique(artifact_column)

This shows all distinct values present in each metadata column.
This is extremely useful because it helps us verify:
  -> what modalities exist
  -> what class labels exist
what artifact values exist


As per the output:

1. Signal shape : (639, 24576)
This confirms: 639 total trials; 24576 EEG values per trial -->> That matches the expected format.

2. Modality values : [1. 2.]
This means there are 2 modality types in this subject file.
From your earlier implementation, we are using:
modality == 1 -> imagined speech
So we will keep only those rows.

3. Stimulus values : [1. 2. 3. 4. 5. 6. 7. 8. 9. 10. 11.]
This means:
there are 11 classes
labels currently run from 1 to 11
Later, for the model, we will convert them to:  0 to 10
because neural networks usually prefer zero-based labels.

4. Artifact values : [1. 2.]
This means the artifact column has 2 possible values. we are currently using:
artifact == 1
as the valid trial condition.
So for now, we will stay consistent with your original implementation.

"""

In [35]:
# ---------------------------------------
# step 8: filtering only imagined speech and valid trials
# ---------------------------------------

valid_trial_mask = (modality_column == 1) & (artifact_column == 1)

filtered_signal_data = signal_data[valid_trial_mask]
filtered_labels = stimulus_column[valid_trial_mask]

print("==============================================")
print("Number of valid filtered trials:", len(filtered_labels))
print("Filtered signal shape:", filtered_signal_data.shape)
print("Filtered labels shape:", filtered_labels.shape)
print("==============================================")

print("Unique filtered labels:", np.unique(filtered_labels))
print("==============================================")

Number of valid filtered trials: 351
Filtered signal shape: (351, 24576)
Filtered labels shape: (351,)
Unique filtered labels: [ 1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11.]


In [36]:
"""
valid_trial_mask = (modality_column == 1) & (artifact_column == 1)
-> This creates a True/False filter.
-> It is asking for each row:
      Is modality equal to 1?
      And is artifact equal to 1?
-> If both are true, keep that row. If not, reject it.
So keeping only the rows that match the research condition.

filtered_signal_data = signal_data[valid_trial_mask]
This keeps only the signal rows that passed the filter.

filtered_labels = stimulus_column[valid_trial_mask]
This keeps only the labels for those same filtered rows.
That way:
signal and labels stay aligned correctly


As per the output:

After filtering:
total valid trials = 351
each valid trial still has 24576 signal values
labels are still present for all 11 classes

Subject S14 gives us 351 usable imagined-speech clean trials


Now we convert the filtered signal from a flat row format into the real EEG trial format:

From this:    (351, 24576)
To this:      (351, 6, 4096)

Because each trial is not just one long line.
It is actually: 6 EEG channels and 4096 samples in each channel
So we now reshape the data back into its meaningful structure.



"""

'\nvalid_trial_mask = (modality_column == 1) & (artifact_column == 1)\n-> This creates a True/False filter.\n-> It is asking for each row: \n      Is modality equal to 1?\n      And is artifact equal to 1?\n-> If both are true, keep that row. If not, reject it.\nSo keeping only the rows that match the research condition.\n\nfiltered_signal_data = signal_data[valid_trial_mask]\nThis keeps only the signal rows that passed the filter.\n\nfiltered_labels = stimulus_column[valid_trial_mask]\nThis keeps only the labels for those same filtered rows.\nThat way:\nsignal and labels stay aligned correctly\n\n\nAs per the output:\n\nAfter filtering:\ntotal valid trials = 351\neach valid trial still has 24576 signal values\nlabels are still present for all 11 classes\n\nSubject S14 gives us 351 usable imagined-speech clean trials\n\n\nNow we convert the filtered signal from a flat row format into the real EEG trial format:\n\nFrom this:    (351, 24576)\nTo this:      (351, 6, 4096)\n\nBecause each 

In [37]:
# ---------------------------------------
# step 9: reshape filtered signal into trial format
# ---------------------------------------


number_of_channels = 6
number_of_samples_per_trial = 4096

X = filtered_signal_data.reshape(-1, number_of_channels, number_of_samples_per_trial)
y = filtered_labels.astype(int)

print("==============================================")
print("X shape after reshape:", X.shape)
print("y shape:", y.shape)
print("==============================================")

X shape after reshape: (351, 6, 4096)
y shape: (351,)


In [40]:
"""
number_of_channels = 6 -->> means, each trial has 6 EEG channels
number_of_samples_per_trial = 4096 -->> each channel has 4096 time samples

filtered_signal_data.reshape(-1, 6, 4096) -->> this means:
take the long flat signal row and reorganize it into (trial, channel, sample)

here, -1 means:  how many trials there are
Since we already have 351 filtered rows, it should become: (351, 6, 4096)


y = filtered_labels.astype(int)
This converts the labels from decimal-looking numbers like:
1.0
2.0
into integer form:
1
2
That is cleaner for classification work.




As per the output:

this means, the filtered data is now in the right EEG trial format:

351 trials
6 channels
4096 samples per trial

So now the data is no longer a flat table.
It is in the proper shape the model can eventually use.


NEXT:

Now we prepare the labels properly.
Right now labels are: 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11

But for deep learning classification, it is cleaner to use:
0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10

So we will shift every label down by 1.

We will also check:

  - how many classes we have
  - how many samples are in each class
This is important because later, when we make the 5 folds, we want the class balance to stay reasonable.

"""

'\nnumber_of_channels = 6 -->> means, each trial has 6 EEG channels\nnumber_of_samples_per_trial = 4096 -->> each channel has 4096 time samples\n\nfiltered_signal_data.reshape(-1, 6, 4096) -->> this means: \ntake the long flat signal row and reorganize it into (trial, channel, sample)\n\nhere, -1 means:  how many trials there are\nSince we already have 351 filtered rows, it should become: (351, 6, 4096)\n\n\ny = filtered_labels.astype(int)\nThis converts the labels from decimal-looking numbers like:\n1.0\n2.0\ninto integer form:\n1\n2\nThat is cleaner for classification work.\n\n\n\n\nAs per the output:\n\nthis means, the filtered data is now in the right EEG trial format:\n\n351 trials\n6 channels\n4096 samples per trial\n\nSo now the data is no longer a flat table.\nIt is in the proper shape the model can eventually use.\n\n'

In [41]:
# ---------------------------------------
# step 10 : prepare labels for classification
# ---------------------------------------

y = y - 1

number_of_classes = len(np.unique(y))
label_counts = pd.Series(y).value_counts().sort_index()

print("==============================================")
print("Unique labels after shifting to 0-based indexing:")
print(np.unique(y))
print("==============================================")

print("Number of classes:", number_of_classes)
print("==============================================")

print("Class counts:")
print(label_counts)
print("==============================================")

Unique labels after shifting to 0-based indexing:
[ 0  1  2  3  4  5  6  7  8  9 10]
Number of classes: 11
Class counts:
0     35
1     40
2     37
3     33
4     34
5     28
6     32
7     31
8     27
9     25
10    29
Name: count, dtype: int64


In [ ]:
"""
y = y - 1 --> this subtracts 1 from every label.
So:
1 becomes 0
2 becomes 1
...
11 becomes 10
Why because most classification pipelines work more naturally when classes start at 0.

number_of_classes = len(np.unique(y))
This counts how many unique class labels exist.

Since the data has labels from 0 to 10, this should become:
11 classes

label_counts = pd.Series(y).value_counts().sort_index()
This counts how many trials belong to each class.

this useful because later,
for 5-fold cross-validation,
-> we want to know whether each class has enough samples and whether the distribution is reasonably balanced.



As per the output:

labels are now prepared correctly:

classes run from 0 to 10
total number of classes is 11
every class has a reasonable number of trials

That is important because 5-fold stratified splitting needs enough samples in each class.
Here,  smallest class has 25 trials, so this is fine for 5 folds.


NEXT:


Now we build the 5-fold split.
This is the main backbone of the experiment.

What 5-fold split means in simple words

  -> we have 351 trials.
  -> We will divide them into 5 parts.

Then:
  -> use 4 parts for training
  -> use 1 part for testing
Repeat this 5 times, so each part gets a turn as the test set.

This gives a more trustworthy result than doing only one random split.

"""

In [42]:
# ---------------------------------------
# step 11: creating 5-fold stratified cross-validation splits
# ---------------------------------------

from sklearn.model_selection import StratifiedKFold

number_of_folds = 5
random_seed = 42

stratified_kfold = StratifiedKFold(
    n_splits=number_of_folds,
    shuffle=True,
    random_state=random_seed
)

fold_splits = list(stratified_kfold.split(X, y))

print("==============================================")
print("Total number of folds created:", len(fold_splits))
print("==============================================")

for fold_number, (train_indices, test_indices) in enumerate(fold_splits, start=1):
    print("Fold", fold_number)
    print("Train size:", len(train_indices))
    print("Test size:", len(test_indices))
    print("----------------------------------------------")


Total number of folds created: 5
Fold 1
Train size: 280
Test size: 71
----------------------------------------------
Fold 2
Train size: 281
Test size: 70
----------------------------------------------
Fold 3
Train size: 281
Test size: 70
----------------------------------------------
Fold 4
Train size: 281
Test size: 70
----------------------------------------------
Fold 5
Train size: 281
Test size: 70
----------------------------------------------


In [ ]:
"""
StratifiedKFold(n_splits=number_of_folds,
    shuffle=True, random_state=random_seed)
 -> This creates a special type of split.

“Stratified” means:
 -> it tries to keep the class balance similar in every fold
So if one class is smaller, it still tries to spread it fairly across the 5 folds.
That is better than a plain random split.

n_splits=5 --> tells to divide the data into 5 folds
shuffle=True --> tells to shuffle the data before splitting
This is useful because it avoids folds being formed in a rigid original order.

random_state=42 --> this is a seed value.
This fixes the randomness so the split is reproducible.
That means if you rerun it, you should get the same folds again.

fold_splits = list(stratified_kfold.split(X, y)) --> this actually creates the folds.

For each fold, it gives two lists:
    1) training indices
    2) testing indices
These are just position numbers telling us which trials belong to train and test.

The FOR loop prints:
  -> fold number
  -> how many training trials
  -> how many testing trials
This is just a check to confirm that the split was created properly.


As per the output:

we have successfully created:
  -> 5 folds
  -> each fold has about 280 training samples
  -> each fold has about 70 test samples

the data is loaded, cleaned, reshaped, labels prepared, and split properly for Phase 1
That is a strong foundation.

NEXT:

first model step: At this step, we are only defining the model.

We are not training it yet.
We are just defining its architecture.

Because in 5-fold cross-validation, we must create a fresh new model for each fold.
So first we define the architecture as a function.


EEGNet is a compact CNN made for EEG signals.
this model tries to:
    1. look for useful patterns across time
    2. combine channel-wise information
    3. reduce the feature size gradually
    4. finally classify into one of the 11 classes
 this is the standard baseline model, and later every improved method will be compared against it


"""

In [44]:
# ---------------------------------------
# step 12 : define baseline EEGNet model
# ---------------------------------------


def build_baseline_eegnet(input_shape, number_of_classes):
    input_layer = Input(shape=input_shape)

    block_1 = Conv2D(
        filters=8,
        kernel_size=(1, 64),
        padding="same",
        use_bias=False
    )(input_layer)
    block_1 = BatchNormalization()(block_1)

    block_1 = DepthwiseConv2D(
        kernel_size=(input_shape[0], 1),
        depth_multiplier=2,
        use_bias=False,
        depthwise_constraint=max_norm(1.0)
    )(block_1)
    block_1 = BatchNormalization()(block_1)
    block_1 = Activation("elu")(block_1)
    block_1 = AveragePooling2D(pool_size=(1, 4))(block_1)
    block_1 = Dropout(0.5)(block_1)

    block_2 = SeparableConv2D(
        filters=16,
        kernel_size=(1, 16),
        padding="same",
        use_bias=False
    )(block_1)
    block_2 = BatchNormalization()(block_2)
    block_2 = Activation("elu")(block_2)
    block_2 = AveragePooling2D(pool_size=(1, 8))(block_2)
    block_2 = Dropout(0.5)(block_2)

    flatten_layer = Flatten()(block_2)

    output_layer = Dense(
        units=number_of_classes,
        activation="softmax",
        kernel_constraint=max_norm(0.25)
    )(flatten_layer)

    model = Model(inputs=input_layer, outputs=output_layer)

    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [ ]:
"""
def build_baseline_eegnet(input_shape, number_of_classes):  -->> This creates a function.
A function is like a machine template.
Whenever we call this function, it will create a fresh EEGNet model.

this is important because, in 5-fold cross-validation, we must create a new model for each fold.
in fold 1, fold 2, fold 3, etc., we must not reuse the same old trained model.
Each fold must start clean.

Input(shape=input_shape) -->> This defines the input shape.

Later our input shape will be:

6 channels
4096 samples
1 extra final dimension for CNN format
So later the model will expect data shaped like: (6, 4096, 1)


Conv2D(...) --> This is the first feature extraction layer.
it starts scanning the EEG trial to find useful time-related patterns

BatchNormalization() --> This helps stabilize the learning process.
it helps keep the internal values more controlled during training

DepthwiseConv2D(...) --> This is important in EEGNet.
this helps the model learn channel-related spatial patterns
Since EEG has multiple channels, this step helps the model learn relationships across channels.

Activation("elu") --> This adds non-linearity.
it helps the model learn more complex patterns, not just straight-line relationships

AveragePooling2D(...) -->   This reduces the size of the feature map.
it compresses information so the model becomes lighter and focuses on the more important patterns

Dropout(0.5) --> This is a very important layer.
during training, it randomly ignores some information so the model does not become too dependent on a few patterns
This helps reduce overfitting.
Since the  dataset is small, dropout is important.

SeparableConv2D(...) --> This is another efficient feature extraction layer.
it learns a second stage of useful patterns, but in a compact way

Flatten() --> This takes the learned feature maps and turns them into one long vector.
it prepares the learned information for the final classifier layer

Dense(..., activation="softmax") --> This is the final classification layer.

In this case:

number_of_classes = 11
So the output will be 11 probabilities.

The model will say:
“I think this trial belongs to class 0, or 1, or 2, ... or 10”

and softmax makes those into class probabilities.

model.compile(...) --> This tells TensorFlow:
    -> what optimizer to use → adam
    -> what loss to use → categorical_crossentropy
    -> what metric to track → accuracy

This prepares the model for training.



NEXT:

Creating the one-fold training function

That function will do all of this for one fold:

take train and test indices
create train / validation / test sets
normalize using training data only
add the last dimension
one-hot encode labels
build a fresh EEGNet
train it
test it
return the fold accuracy

That is the most important coding step in Phase 1.

"""

In [46]:
# ---------------------------------------
# Step 14: function to train and evaluate one fold
# ---------------------------------------

def run_one_fold(X, y, train_indices, test_indices, fold_number):
    # ------------------------------------------
    # Step 14.1: create training and test data
    # ------------------------------------------
    X_train_full = X[train_indices]
    y_train_full = y[train_indices]

    X_test = X[test_indices]
    y_test = y[test_indices]

    # ------------------------------------------
    # Step 14.2: create validation set only from training data
    # ------------------------------------------
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full,
        y_train_full,
        test_size=validation_size_inside_training,
        stratify=y_train_full,
        random_state=random_seed
    )

    # ------------------------------------------
    # Step 14.3: normalize using only training data
    # ------------------------------------------
    training_mean = np.mean(X_train)
    training_std = np.std(X_train)

    if training_std == 0:
        training_std = 1.0

    X_train = (X_train - training_mean) / training_std
    X_val = (X_val - training_mean) / training_std
    X_test = (X_test - training_mean) / training_std

    # ------------------------------------------
    # Step 14.4: add final dimension for CNN input
    # ------------------------------------------
    X_train = X_train[..., np.newaxis]
    X_val = X_val[..., np.newaxis]
    X_test = X_test[..., np.newaxis]

    # ------------------------------------------
    # Step 14.5: convert labels to one-hot format
    # ------------------------------------------
    y_train_one_hot = to_categorical(y_train, num_classes=number_of_classes)
    y_val_one_hot = to_categorical(y_val, num_classes=number_of_classes)
    y_test_one_hot = to_categorical(y_test, num_classes=number_of_classes)

    # ------------------------------------------
    # Step 14.6: build a fresh model
    # ------------------------------------------
    input_shape = (number_of_channels, number_of_samples_per_trial, 1)

    model = build_baseline_eegnet(
        input_shape=input_shape,
        number_of_classes=number_of_classes
    )

    print("==============================================")
    print("Running fold number:", fold_number)
    print("Training shape:", X_train.shape)
    print("Validation shape:", X_val.shape)
    print("Test shape:", X_test.shape)
    print("==============================================")

    # ------------------------------------------
    # Step 14.7: train the model
    # ------------------------------------------
    history = model.fit(
        X_train,
        y_train_one_hot,
        epochs=number_of_epochs,
        batch_size=batch_size,
        validation_data=(X_val, y_val_one_hot),
        verbose=1
    )

    # ------------------------------------------
    # Step 14.8: evaluate on the test fold
    # ------------------------------------------
    test_loss, test_accuracy = model.evaluate(
        X_test,
        y_test_one_hot,
        verbose=0
    )

    print("Fold", fold_number, "test accuracy:", test_accuracy)
    print("==============================================")

    result_dictionary = {
        "fold_number": fold_number,
        "train_size": int(len(X_train)),
        "validation_size": int(len(X_val)),
        "test_size": int(len(X_test)),
        "test_loss": float(test_loss),
        "test_accuracy": float(test_accuracy)
    }

    return result_dictionary, history



In [ ]:
"""
This function will do the full work for one fold only.

It will:

take one fold’s train and test indices
create training and test datasets
create a validation set from the training data only
normalize using training data only
add the final CNN dimension
convert labels to one-hot format
build a fresh baseline EEGNet
train it
test it
return the result


UNDERSTANDING THE LOGIC:

In 5-fold cross-validation, each fold must be treated like a fresh experiment.

So for fold 1:

use fold 1 test set only for final testing
use the remaining data for training
from training data, carve out a small validation set

Then for fold 2:

build a completely fresh model again
do not continue from fold 1 model

This is very important.



Part 1 — Training and test split for the current fold:

X_train_full = X[train_indices]
y_train_full = y[train_indices]

X_test = X[test_indices]
y_test = y[test_indices]

This takes the indices from one fold and separates:

training part
testing part

At this stage:

test set is untouched
training set still needs a validation split





Part 2 — Validation split from training only
train_test_split(... test_size=validation_size_inside_training ...)

This creates:

training subset
validation subset

Important:
this validation set comes only from training data, not from test data.

That is the correct way.

Part 3 — Normalize using training data only
training_mean = np.mean(X_train)
training_std = np.std(X_train)

This is very important for research quality.

We calculate:

mean
standard deviation

using only the training data

Then we use those same values to normalize:

training data
validation data
test data

Why is this correct?

Because the test data should not influence the normalization statistics.

Part 4 — Add the extra last dimension
X_train = X_train[..., np.newaxis]

This changes the shape from:

(samples, 6, 4096)

to:

(samples, 6, 4096, 1)

That final 1 is required because EEGNet uses 2D CNN layers.

Part 5 — One-hot encoding
to_categorical(...)

Your labels are currently:

0 to 10

But the model output is a vector of 11 class probabilities.

So we convert labels into one-hot format.

Example:

class 3 becomes [0,0,0,1,0,0,0,0,0,0,0]

This is the right format for categorical_crossentropy.

Part 6 — Build a fresh model
model = build_baseline_eegnet(...)

This creates a brand new EEGNet for this fold.

Very important:
each fold starts fresh.

Part 7 — Train the model
model.fit(...)

This is where the learning happens.

The model sees:

training data
validation data

and trains for the number of epochs you set.

Part 8 — Evaluate on the test fold
model.evaluate(...)

After training ends, this checks performance on the unseen test fold.

That gives:

test loss
test accuracy
Part 9 — Return results

The function returns:

fold number
train size
validation size
test size
test loss
test accuracy
training history

This is useful because later we will collect fold results from all 5 folds.

What should happen when you run this cell

This cell should normally run with no output, because again it is only defining the function.

That is okay.

Very important note about time

The actual training happens only when we call this function later.

So right now, even if there is no output, that is normal.




"""

In [48]:
# ==============================================
# Step 15: test only Fold 1 first
# ==============================================

first_train_indices, first_test_indices = fold_splits[0]

fold_1_result, fold_1_history = run_one_fold(
    X=X,
    y=y,
    train_indices=first_train_indices,
    test_indices=first_test_indices,
    fold_number=1
)

print("Fold 1 finished.")
print(fold_1_result)

Running fold number: 1
Training shape: (224, 6, 4096, 1)
Validation shape: (56, 6, 4096, 1)
Test shape: (71, 6, 4096, 1)
Epoch 1/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 9s 382ms/step - accuracy: 0.1071 - loss: 2.4433 - val_accuracy: 0.0714 - val_loss: 2.3924
Epoch 2/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 5s 362ms/step - accuracy: 0.6161 - loss: 1.9915 - val_accuracy: 0.1071 - val_loss: 2.3866
Epoch 3/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 7s 520ms/step - accuracy: 0.7411 - loss: 1.7510 - val_accuracy: 0.1250 - val_loss: 2.3846
Epoch 4/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 5s 353ms/step - accuracy: 0.8348 - loss: 1.5599 - val_accuracy: 0.1250 - val_loss: 2.3817
Epoch 5/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 6s 458ms/step - accuracy: 0.8616 - loss: 1.4402 - val_accuracy: 0.1250 - val_loss: 2.3842
Epoch 6/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 9s 351ms/step - accuracy: 0.9062 - loss: 1.3418 - val_accuracy: 0.1429 - val_loss: 2.3904
Epoch 7/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 7s 515ms/step - accuracy: 0.9107 - loss: 1.3189 - val_accuracy: 0.1429 - 

In [ ]:
"""

Line 1
first_train_indices, first_test_indices = fold_splits[0]

This takes the first fold from your 5 prepared folds.

Remember:

fold_splits contains 5 train/test index pairs
[0] means the first one

So now:

first_train_indices = training positions for fold 1
first_test_indices = testing positions for fold 1
Line 2 onward
fold_1_result, fold_1_history = run_one_fold(...)

This calls the function you just created.

That function will now do all the actual work:

build train/test data
make validation split
normalize correctly
add CNN dimension
one-hot encode labels
build fresh EEGNet
train the model
test the model
return fold result
Last lines
print("Fold 1 finished.")
print(fold_1_result)

This prints the result dictionary.

You should see something containing:

fold number
train size
validation size
test size
test loss
test accuracy


The full pipeline successfully completed for one fold:

data split worked
validation split worked
normalization worked
CNN input shape worked
one-hot labels worked
baseline EEGNet trained
test evaluation worked

So the Phase 1 pipeline is now functionally correct for one fold.

As per the output:

Training shape: (224, 6, 4096, 1)
Validation shape: (56, 6, 4096, 1)
Test shape: (71, 6, 4096, 1)

This is correct. Because Fold 1 originally had:

280 training-side samples
71 test samples

Then from the 280 training-side samples, you kept:

224 for training
56 for validation

That is exactly right.

Training behavior

Your training accuracy rose very high:

it started low
then climbed toward 99%

But the validation accuracy stayed low:

around 10% to 18%, sometimes near 19.6%

And final test accuracy was:

0.2112676054239273
which is about 21.13%
What this means in simple words

This is showing the same pattern your professor warned about:

The model is overfitting

Meaning:

on training data, it becomes very strong
on new unseen data, it does not generalize well

That is why:

training accuracy is near 99%
validation is much lower
test is only around 21%

This is not a failure.

This is actually expected in your project because:

dataset is small
model still has enough power to memorize
imagined speech EEG is hard

So this result is scientifically meaningful.

It confirms that:

your baseline pipeline works
overfitting is real
Professor Rufiner’s concern is valid
the next later stages (augmentation, transfer learning, attention) are necessary
Why 21.13% is still meaningful

You have 11 classes.

So random guessing would be around:

1 / 11 = 0.0909

That is about 9.09%

Your Fold 1 test accuracy is about 21.13%, which is above random chance.

So even though the model overfits, it is still learning something useful.

That is important.

"""

In [49]:
# ---------------------------------------
# step  16 : run all 5 folds
# ---------------------------------------

all_fold_results = []
all_fold_histories = []

for fold_number, (train_indices, test_indices) in enumerate(fold_splits, start=1):
    fold_result, fold_history = run_one_fold(
        X=X,
        y=y,
        train_indices=train_indices,
        test_indices=test_indices,
        fold_number=fold_number
    )

    all_fold_results.append(fold_result)
    all_fold_histories.append(fold_history)

print("==============================================")
print("All 5 folds finished.")
print("==============================================")


Running fold number: 1
Training shape: (224, 6, 4096, 1)
Validation shape: (56, 6, 4096, 1)
Test shape: (71, 6, 4096, 1)
Epoch 1/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 14s 385ms/step - accuracy: 0.0759 - loss: 2.4591 - val_accuracy: 0.0714 - val_loss: 2.3914
Epoch 2/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 5s 388ms/step - accuracy: 0.5268 - loss: 1.9895 - val_accuracy: 0.1071 - val_loss: 2.3846
Epoch 3/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 7s 467ms/step - accuracy: 0.7054 - loss: 1.7067 - val_accuracy: 0.0893 - val_loss: 2.3767
Epoch 4/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 10s 440ms/step - accuracy: 0.8170 - loss: 1.5137 - val_accuracy: 0.1071 - val_loss: 2.3740
Epoch 5/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 6s 404ms/step - accuracy: 0.8571 - loss: 1.3729 - val_accuracy: 0.0893 - val_loss: 2.3759
Epoch 6/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 12s 530ms/step - accuracy: 0.8839 - loss: 1.3285 - val_accuracy: 0.0893 - val_loss: 2.3710
Epoch 7/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 5s 354ms/step - accuracy: 0.9062 - loss: 1.2920 - val_accuracy: 0.0357

In [ ]:
"""
What this code does

This loops through all 5 folds.

For each fold, it:

gets that fold’s train indices
gets that fold’s test indices
runs the full training pipeline
saves the fold result
saves the training history

At the end:

all_fold_results will hold the 5 fold results
all_fold_histories will hold the 5 training histories
Important note before you run it

Since you already ran Fold 1 separately, this new loop will run Fold 1 again as part of the full 5-fold process.

That is okay.

That is actually better, because then all 5 folds are generated in one consistent run.

So do not worry that Fold 1 will repeat.

It may take time

This step will take much longer than Fold 1 alone, because now it will train 5 times.

That is normal.






"""

In [50]:

# ==============================================
# Step 17: convert all fold results into a table
# ==============================================

results_dataframe = pd.DataFrame(all_fold_results)

print("==============================================")
print("Fold-wise results table:")
print("==============================================")
display(results_dataframe)


Fold-wise results table:


,fold_number,train_size,validation_size,test_size,test_loss,test_accuracy
0,1,224,56,71,2.439090,0.126761
1,2,224,57,70,2.481131,0.142857
2,3,224,57,70,2.641555,0.085714
3,4,224,57,70,2.637326,0.157143
4,5,224,57,70,2.497113,0.128571


In [ ]:
"""


pd.DataFrame(all_fold_results)

This takes the 5 result dictionaries and turns them into a neat table.

So instead of seeing results as scattered Python objects, you will see rows like:

fold number
train size
validation size
test size
test loss
test accuracy
display(results_dataframe)

This shows the table nicely inside Colab.


"""

In [51]:

# ==============================================
# Step 18: calculate final mean and standard deviation
# ==============================================

mean_accuracy = results_dataframe["test_accuracy"].mean()
std_accuracy = results_dataframe["test_accuracy"].std()

print("==============================================")
print("Final baseline EEGNet result for this subject")
print("==============================================")
print("Subject file name:", subject_file_name)
print("Fold accuracies:", results_dataframe["test_accuracy"].tolist())
print("Mean accuracy:", mean_accuracy)
print("Standard deviation:", std_accuracy)
print("Mean ± Std:", str(round(mean_accuracy, 4)) + " ± " + str(round(std_accuracy, 4)))
print("==============================================")

Final baseline EEGNet result for this subject
Subject file name: S14_EEG.mat
Fold accuracies: [0.1267605572938919, 0.1428571492433548, 0.08571428805589676, 0.15714286267757416, 0.12857143580913544]
Mean accuracy: 0.1282092586159706
Standard deviation: 0.026738392689945087
Mean ± Std: 0.1282 ± 0.0267


In [ ]:
"""
What this code does
mean()

This gives the average test accuracy across all 5 folds.

That is your main baseline result for this subject.

std()

This gives the standard deviation.

That tells you how stable or unstable the model is across folds.

Simple meaning:
low standard deviation = more stable
high standard deviation = more variation across folds
results_dataframe["test_accuracy"].tolist()

This prints the five fold accuracies as a list.

That is useful because in a paper or report, you often want:

the individual fold values
and the final mean ± std
Why this step matters

Now your result is no longer:

“one lucky split gave one number”

Now it becomes:

“subject-wise 5-fold cross-validation gave a mean result with variability”

That is far stronger scientifically




"""

In [52]:

# ==============================================
# Optional Step 19: save results to Google Drive
# ==============================================

import json

os.makedirs(results_folder, exist_ok=True)

csv_output_path = os.path.join(
    results_folder,
    subject_file_name.replace(".mat", "_baseline_5fold_results.csv")
)

json_output_path = os.path.join(
    results_folder,
    subject_file_name.replace(".mat", "_baseline_5fold_summary.json")
)

results_dataframe.to_csv(csv_output_path, index=False)

summary_dictionary = {
    "subject_file_name": subject_file_name,
    "number_of_folds": number_of_folds,
    "number_of_classes": int(number_of_classes),
    "mean_accuracy": float(mean_accuracy),
    "std_accuracy": float(std_accuracy),
    "fold_accuracies": results_dataframe["test_accuracy"].tolist()
}

with open(json_output_path, "w") as json_file:
    json.dump(summary_dictionary, json_file, indent=4)

print("==============================================")
print("Results saved successfully.")
print("CSV path:", csv_output_path)
print("JSON path:", json_output_path)
print("==============================================")

Results saved successfully.
CSV path: /content/drive/MyDrive/Colab_Notebooks/My_Project/Results_April2026/S14_EEG_baseline_5fold_results.csv
JSON path: /content/drive/MyDrive/Colab_Notebooks/My_Project/Results_April2026/S14_EEG_baseline_5fold_summary.json


In [ ]:
"""

Phase 1 — Baseline EEGNet (Subject-wise 5-Fold CV)

Now let’s understand what your result actually means (very important for your paper + viva).

Final Result

We got:

Fold accuracies: [0.1267, 0.1429, 0.0857, 0.1571, 0.1286]

Mean accuracy: 0.1282  -> 12.82%

Standard deviation: 0.0267  ->   ~2.67%

What this means
1. Compare with random guessing
You have 11 classes.
So random guessing = 1 / 11 ≈ 9.09%
model = 12.82%
-->> Slightly better than random
-->> Not strong performance

2. Overfitting is clearly visible
From  earlier training logs:

Training accuracy → ~95% to 99%
Validation accuracy → ~10–18%
Test accuracy → ~12%
This means:
The model is memorizing training data, but not generalizing
This is exactly what your professor warned about.

3. Standard deviation (~2.6%)
This tells us:
Results across folds are consistent
Not wildly fluctuating

That’s good for research
How to describe this in  report

“Baseline EEGNet achieved a mean classification accuracy of 12.82%
with a standard deviation of 2.67% under subject-specific 5-fold cross-validation.
Although performance is slightly above chance level (~9.09%),
the model exhibits severe overfitting, as evidenced by high training accuracy and low validation/test performance.”


With this experiment, we have scientifically shown:

1. Baseline EEGNet is not enough
✔ Confirmed

2. Data is too small for the model
✔ Confirmed

3. Overfitting is real and severe
✔ Confirmed

4. Professor Rufiner’s feedback is correct
✔ Confirmed




NEXT: Phase 2 — Attention Model (Subject-wise 5-Fold CV)
EEGNet → EEGNet + Attention


"""

In [ ]:
# ---------------------------------------
#
# ---------------------------------------

In [ ]:
# ---------------------------------------
#
# ---------------------------------------